In [1]:
import sqlite3, pandas as pd, os

RUN_ID = "e6ddf80e-52a5-493a-9f69-8bbf2b994387"
db_path = os.path.abspath("../data/target.db")
conn = sqlite3.connect(db_path)

def q(name, sql, params=None):
    print(f"\n--- {name} ---")
    df = pd.read_sql(sql, conn, params=params)
    display(df)
    return df

# 1) Run summary
q("migration_runs row", "SELECT * FROM migration_runs WHERE run_id = ?", (RUN_ID,))

# 2) True reconciliation: raw = clean + rejected
q("reconciliation", """
SELECT
  total_raw, total_clean, total_rejected,
  (total_clean + total_rejected) AS clean_plus_rejected,
  CASE WHEN total_raw = (total_clean + total_rejected) THEN 'PASS' ELSE 'FAIL' END AS status
FROM migration_runs
WHERE run_id = ?;
""", (RUN_ID,))

# 3) Raw duplicates (expected)
q("raw duplicate customer_ids (top 20)", """
SELECT customer_id, COUNT(*) AS cnt
FROM legacy_customers_raw
WHERE run_id = ?
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20;
""", (RUN_ID,))

# 4) Target duplicates (must be 0)
q("clean duplicate customer_ids (must be empty)", """
SELECT customer_id, COUNT(*) AS cnt
FROM customers_clean
WHERE run_id = ?
GROUP BY customer_id
HAVING COUNT(*) > 1;
""", (RUN_ID,))

# 5) Compliance: ACTIVE must be VERIFIED (must be 0)
q("ACTIVE without VERIFIED KYC (must be empty)", """
SELECT *
FROM customers_clean
WHERE run_id = ?
  AND account_status = 'ACTIVE'
  AND kyc_status <> 'VERIFIED';
""", (RUN_ID,))

# 6) Financial integrity: negative balances (must be 0)
q("negative balances (must be empty)", """
SELECT *
FROM customers_clean
WHERE run_id = ?
  AND balance < 0;
""", (RUN_ID,))

# 7) Currency integrity (must be 0)
q("unsupported currency (must be empty)", """
SELECT *
FROM customers_clean
WHERE run_id = ?
  AND currency IS NOT NULL
  AND currency NOT IN ('USD','INR');
""", (RUN_ID,))

# 8) Top reject reasons (audit)
q("top reject reasons", """
SELECT reject_reason, COUNT(*) AS cnt
FROM customers_rejected
WHERE run_id = ?
GROUP BY reject_reason
ORDER BY cnt DESC
LIMIT 20;
""", (RUN_ID,))

conn.close()



--- migration_runs row ---


,run_id,start_time,end_time,source_file,total_raw,total_clean,total_rejected,status,notes
0,e6ddf80e-52a5-493a-9f69-8bbf2b994387,2026-02-11T19:13:32Z,2026-02-11T19:13:48Z,data/legacy_customers.csv,1050,39,1011,SUCCESS,Load completed



--- reconciliation ---


,total_raw,total_clean,total_rejected,clean_plus_rejected,status
0,1050,39,1011,1050,PASS



--- raw duplicate customer_ids (top 20) ---


...

,run_id,customer_id,full_name,email,country,signup_date,credit_score,account_status,kyc_status,last_updated,balance,currency



--- unsupported currency (must be empty) ---


,run_id,customer_id,full_name,email,country,signup_date,credit_score,account_status,kyc_status,last_updated,balance,currency



--- top reject reasons ---


,reject_reason,cnt
0,invalid_signup_date;invalid_last_updated,155
1,invalid_signup_date,106
2,invalid_last_updated,44
3,invalid_signup_date;invalid_last_updated;activ...,41
4,invalid_signup_date;active_requires_verified_kyc,37
5,invalid_signup_date;missing_account_status,33
6,invalid_signup_date;invalid_last_updated;missi...,33
7,invalid_signup_date;invalid_last_updated;missi...,25
8,invalid_signup_date;missing_kyc_status,24
9,invalid_signup_date;invalid_last_updated;inval...,22
